In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
movies = pd.read_csv("../data/processed_movies.csv")
display(movies.head())

,movieId,title,genres,clean_title,tag,metadata
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,Toy Story,children Disney animation children Disney Disn...,Toy Story Adventure Animation Children Comedy...
1,2,Jumanji (1995),Adventure Children Fantasy,Jumanji,Robin Williams fantasy Robin Williams time tra...,Jumanji Adventure Children Fantasy Robin Will...
2,3,Grumpier Old Men (1995),Comedy Romance,Grumpier Old Men,comedinha de velhinhos engraÃƒÂ§ada comedinha ...,Grumpier Old Men Comedy Romance comedinha de ...
3,4,Waiting to Exhale (1995),Comedy Drama Romance,Waiting to Exhale,characters slurs based on novel or book chick ...,Waiting to Exhale Comedy Drama Romance charac...
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Fantasy pregnancy remake family Steve Martin s...,Father of the Bride Part II Comedy Fantasy pr...


In [4]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['metadata'])

#cos_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [5]:
def recommend(title, num=10, round=4):

    matches = movies[
        movies['title']
        .str.lower()
        .str.contains(title.lower(), regex=False)
    ]

    if matches.empty:
        return "Movie not found."

    title = matches.iloc[0]['title']
    print(f"Recommending for: {title}")
        
    idx = indices[title]

    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores = list(enumerate(sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num+1]

    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]

    result = movies[['movieId', 'title', 'metadata']].iloc[movie_indices].copy()
    result['similarity_score'] = scores
    result['similarity_score'] = result['similarity_score'].round(round)

    result = result.reset_index(drop=True)

    return result

In [6]:
recommend('Toy Story', num=10, round=3)

Recommending for: Toy Story (1995)


,movieId,title,metadata,similarity_score
0,3114,Toy Story 2 (1999),Toy Story 2 Adventure Animation Children Come...,0.888
1,2355,"Bug's Life, A (1998)","Bug's Life, A Adventure Animation Children Co...",0.804
2,78499,Toy Story 3 (2010),Toy Story 3 Adventure Animation Children Come...,0.707
3,4886,"Monsters, Inc. (2001)","Monsters, Inc. Adventure Animation Children C...",0.699
4,157296,Finding Dory (2016),Finding Dory Adventure Animation Comedy point...,0.698
5,6377,Finding Nemo (2003),Finding Nemo Adventure Animation Children Com...,0.693
6,8961,"Incredibles, The (2004)","Incredibles, The Action Adventure Animation C...",0.683
7,103141,Monsters University (2013),Monsters University Adventure Animation Comed...,0.652
8,45517,Cars (2006),Cars Animation Children Comedy animation come...,0.645
9,50872,Ratatouille (2007),Ratatouille Animation Children Drama animatio...,0.637


In [7]:
recommend('Shrek', num=10, round=3)

Recommending for: Shrek (2001)


,movieId,title,metadata,similarity_score
0,8360,Shrek 2 (2004),Shrek 2 Adventure Animation Children Comedy M...,0.862
1,123894,Fairy Tales (1978),Fairy Tales Comedy Romance fairy tale musical...,0.494
2,2096,Sleeping Beauty (1959),Sleeping Beauty Animation Children Musical Di...,0.476
3,42734,Hoodwinked! (2005),Hoodwinked! Animation Children Comedy bad ani...,0.471
4,164659,The Girl Without Hands (2016),The Girl Without Hands Animation fairy tale,0.454
5,205585,It's a Fairy! (2016),It's a Fairy! Comedy,0.448
6,230179,A Fairy Tale Wedding (2014),A Fairy Tale Wedding Comedy,0.444
7,123093,Happily N'Ever After 2 (2009),Happily N'Ever After 2 Animation Children com...,0.441
8,63237,Happily Ever After (1993),Happily Ever After Animation Children Comedy ...,0.440
9,2876,Thumbelina (1994),Thumbelina Animation Children Fantasy innocen...,0.424
